# Test B — continued Qwen3-8B adapter, two optimizer steps

Upload this notebook into Colab or Kaggle. Use a Linux x86-64 notebook runtime (Python 3.10 or newer).
The training subprocess uses Python 3.10–3.12; a newer notebook kernel automatically gets a separate
Python 3.12.11 environment via pinned uv, without changing the notebook kernel or training requirements.
Do not allocate paid hardware until you approve its provider, GPU, actual hourly rate and budget.
Select one 24 GB-class GPU (at least 22 GiB reported total, 20 GiB free), compute capability 7.x–9.x.
Newer compute architectures need a separately reviewed CUDA stack. Two 16 GB GPUs are not a substitute.
On Kaggle enable Internet and expose only GPU 0. If only a T4/P100 16 GB allocation is available, stop.

The original 350.7 MB ZIP is uploaded separately; the cloud cannot access your Mac SSD.
No fresh adapter or new dataset is created. Initial rank-32 RSLoRA is preserved.
The saved legacy adapter has `revision=null`; its original base revision is unproven.
This job explicitly loads Qwen revision `b968826d9c46dd6066d109eabc6255188de91218`.

Cells 1–4 install/verify dependencies and load only a tokenizer. Cell 5 requires typed authorization
before any full-model loading or training. Cell 6 exports results or failure diagnostics.
There is no inference/model-quality evaluation or checkpoint promotion beyond one nonempty-generation smoke test.


## 1. Initialize the transfer workspace and embedded runner


In [ ]:
import os, sys, json, subprocess, shutil, hashlib, platform
from pathlib import Path
host = {'system': platform.system(), 'machine': platform.machine(), 'python': platform.python_version()}
print('Notebook host:', json.dumps(host))
if host['system'] != 'Linux' or host['machine'] != 'x86_64' or sys.version_info[:2] < (3, 10):
    raise RuntimeError(f'Need Linux x86_64 and a Python >=3.10 notebook kernel; detected {host}')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
BASE = Path('/kaggle/working/test-b') if Path('/kaggle/working').exists() else Path('/content/test-b')
BASE.mkdir(parents=True, exist_ok=True)
ROOT = BASE / 'package'
RUNNER = BASE / 'gpu_canary_runner.py'
VENV = BASE / 'venv'
PY = VENV / 'bin/python'
os.environ['HF_HOME'] = str(BASE / 'hf-cache')

def execute(command, log_name):
    log = BASE / log_name
    if log.exists():
        raise RuntimeError(f'Log already exists: {log}. Inspect earlier attempt before rerunning.')
    with log.open('w') as stream:
        process = subprocess.Popen([str(x) for x in command], cwd=BASE, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                stream.write(line); stream.flush(); print(line, end='')
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if returncode:
        raise RuntimeError(f'Command failed ({returncode}); inspect {log}. Do not continue to training.')


In [ ]:
RUNNER.write_text('"""Observability and authorization wrapper for the immutable Strategy A package.\n\nThis file stays OUTSIDE the checksummed package. It never changes training data,\nmodel settings, or the packaged trainer. Free resources use the same worker core\nwith a wall-clock watchdog because the old paid CLI rejects a zero hourly rate.\n"""\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport signal\nimport sys\nimport time\nimport traceback\nimport zipfile\n\nJOB = \'c03c14696b8ab4b394266f83d850641ae3e71f9eaa3cc749a34de12489834fc5\'\nARCHIVE_SHA = \'19ec9e08f6bd85eb7ee585dbb62939db7c3fbcaa588f70f853b89b82a5751e2d\'\nREVISION = \'b968826d9c46dd6066d109eabc6255188de91218\'\nDATASET = \'51cb0864daa7680acd3afd5d68d31276bac2289cc24bf7921cbc850e81252810\'\nADAPTER = {\n    \'adapter_config.json\': \'1fe1fdbfa6b611c63f9d7dc84d431f2b3685338d91678510f579b24ae002d5df\',\n    \'adapter_model.safetensors\': \'154c508775106e8ffe15e224c9f256131852d384e4efaea6af2bdf3d35e99c6f\',\n}\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef save(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\')\n    tmp.replace(path)\n\n\ndef extract(archive, root):\n    if sha(archive) != ARCHIVE_SHA:\n        raise ValueError(\'Uploaded archive SHA-256 mismatch\')\n    if root.exists():\n        raise ValueError(\'Extraction directory already exists; preserve previous attempts\')\n    with zipfile.ZipFile(archive) as z:\n        names = z.namelist()\n        if len(names) != len(set(names)):\n            raise ValueError(\'Duplicate archive members\')\n        for entry in z.infolist():\n            target = root / entry.filename\n            mode = entry.external_attr >> 16\n            if not target.resolve().is_relative_to(root.resolve()) or mode & 0o170000 == 0o120000:\n                raise ValueError(\'Unsafe ZIP member\')\n        z.extractall(root)\n    return verify(root)\n\n\ndef verify(root):\n    sys.path.insert(0, str(root))\n    from autonomous_learning.jobs import verify_package, adapter_compatibility\n    from autonomous_learning.dataset import validate_example\n    job = verify_package(root)\n    if (job[\'job_id\'] != JOB or job[\'dataset_id\'] != DATASET\n            or job[\'model_name\'] != \'Qwen/Qwen3-8B\' or job[\'model_revision\'] != REVISION\n            or job[\'adapter_strategy\'] != \'continued\' or job[\'adapter_files\'] != ADAPTER\n            or job[\'output_directory\'] != \'result\' or job[\'planned_max_gpu_usd\'] != 1):\n        raise ValueError(\'Not the approved immutable Strategy A job\')\n    expected = dict(lora_rank=32, lora_alpha=32, lora_dropout=0.0, use_rslora=True,\n                    learning_rate=0.00002, per_device_train_batch_size=1,\n                    gradient_accumulation_steps=4, max_steps_per_round=2,\n                    max_sequence_length=4096)\n    if any(job[\'config\'][\'training\'].get(k) != v for k, v in expected.items()):\n        raise ValueError(\'Strategy A training settings changed\')\n    if not adapter_compatibility(root / \'initial_adapter\', job[\'config\'][\'training\'])[\'compatible\']:\n        raise ValueError(\'Starting adapter is incompatible\')\n    rows = [json.loads(x) for x in (root / \'examples.jsonl\').read_text().splitlines()]\n    if len(rows) != 85 or sum(x[\'replay\'] is True for x in rows) != 21:\n        raise ValueError(\'Expected 64 new examples and 21 replay examples\')\n    for row in rows:\n        validate_example(row)\n    return job\n\n\ndef hardware():\n    import importlib.metadata\n    import platform\n    import subprocess\n    import torch\n    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:\n        raise ValueError(\'Exactly one visible CUDA GPU is required\')\n    props = torch.cuda.get_device_properties(0)\n    free, total = torch.cuda.mem_get_info(0)\n    # Accommodate reported capacity of marketed 24 GB devices, never 16 GB cards.\n    if total < 22 * 1024**3 or free < 20 * 1024**3 or not 7 <= props.major <= 9:\n        raise ValueError(f\'Need a 24 GB-class GPU with >=20 GiB free and CC 7.x–9.x for pinned CUDA 12.6: {props.name}; free={free}, total={total}\')\n    versions = {p: importlib.metadata.version(p) for p in\n                (\'torch\', \'transformers\', \'trl\', \'peft\', \'accelerate\', \'bitsandbytes\', \'datasets\', \'lighteval\', \'safetensors\')}\n    return {\'gpu\': props.name, \'compute_capability\': [props.major, props.minor],\n            \'total_vram_bytes\': total, \'free_vram_bytes\': free, \'cuda_runtime\': torch.version.cuda,\n            \'python\': platform.python_version(), \'platform\': platform.platform(),\n            \'versions\': versions, \'nvidia_smi\': subprocess.check_output([\'nvidia-smi\'], text=True)}\n\n\ndef preflight(root):\n    import subprocess\n    job = verify(root)\n    environment = hardware()\n    from distillation_loop.evaluation import require_training_versions\n    require_training_versions()\n    # A separate process keeps validate-only\'s CPU Accelerate state out of training.\n    completed = subprocess.run([sys.executable, \'-m\', \'autonomous_learning.gpu_worker\', \'.\', \'--validate-only\'],\n                               cwd=root, text=True, capture_output=True)\n    (root.parent / \'validate-only.stdout.log\').write_text(completed.stdout)\n    (root.parent / \'validate-only.stderr.log\').write_text(completed.stderr)\n    if completed.returncode:\n        raise RuntimeError(\'Worker validate-only failed; inspect validate-only.stderr.log\')\n    analysis = json.loads(completed.stdout)\n    if analysis[\'status\'] != \'tokenizer_validated_only\' or analysis[\'examples\'] != 85 or analysis[\'overlength_examples\']:\n        raise ValueError(\'Tokenizer preflight failed\')\n    receipt = {\'status\': \'passed\', \'job_id\': job[\'job_id\'], \'environment\': environment, \'tokenizer\': analysis}\n    save(root.parent / \'cloud-preflight.json\', receipt)\n    return receipt\n\n\ndef authorization(args, environment):\n    if args.billing_mode not in (\'free\', \'paid\'):\n        raise ValueError(\'Confirm whether this specific allocation is free or paid\')\n    if not args.authorize_training or not args.provider or args.expected_gpu != environment[\'gpu\']:\n        raise ValueError(\'Explicit training authorization, provider and exact detected GPU name are required\')\n    if not math.isfinite(args.max_gpu_usd) or not 0 < args.max_gpu_usd <= 1:\n        raise ValueError(\'Budget must be positive and at most the prepared $1 limit\')\n    if not math.isfinite(args.max_runtime_seconds) or not 0 < args.max_runtime_seconds <= 1200:\n        raise ValueError(\'Choose a worker wall-clock limit of at most 1200 seconds\')\n    rate = args.gpu_hourly_rate_usd\n    if not math.isfinite(rate) or rate < 0:\n        raise ValueError(\'Hourly rate must be finite and nonnegative\')\n    if args.billing_mode == \'free\' and rate != 0:\n        raise ValueError(\'Confirmed free resources require actual hourly charge 0\')\n    if args.billing_mode == \'paid\' and rate <= 0:\n        raise ValueError(\'Paid mode requires the actual approved positive hourly rate\')\n\n\ndef train(root, args):\n    job = verify(root)\n    receipt = json.loads((root.parent / \'cloud-preflight.json\').read_text())\n    if receipt[\'status\'] != \'passed\' or receipt[\'job_id\'] != JOB:\n        raise ValueError(\'Run cloud preflight first\')\n    environment = hardware()\n    authorization(args, environment)\n    if (root / \'result\').exists() or (root.parent / \'training-observations.json\').exists():\n        raise ValueError(\'Previous attempt exists; never retrain automatically\')\n    import torch\n    import trl\n    from autonomous_learning import gpu_worker\n    from autonomous_learning.storage import atomic_json\n    from distillation_loop.student import StudentRuntime\n    from distillation_loop.adapter_identity import adapter_files\n    observations = {\'status\': \'started\', \'provider\': args.provider, \'billing_mode\': args.billing_mode,\n                    \'environment\': environment, \'job_id\': JOB, \'dataset_id\': DATASET,\n                    \'approved_max_gpu_usd\': args.max_gpu_usd, \'approved_gpu_hourly_rate_usd\': args.gpu_hourly_rate_usd,\n                    \'max_runtime_seconds\': args.max_runtime_seconds, \'adapter_restores\': [],\n                    \'lineage_limitation\': \'The recovered adapter has revision=null; this continuation uses the pinned job revision.\'}\n    path = root.parent / \'training-observations.json\'\n    save(path, observations)\n    original_train, original_prepare = trl.SFTTrainer.train, StudentRuntime.prepare\n    original_restore, original_infer = StudentRuntime.restore, StudentRuntime.infer\n\n    def observe_prepare(runtime, *a, **kw):\n        value = original_prepare(runtime, *a, **kw)\n        loaded = bool(getattr(runtime.model, \'is_loaded_in_4bit\', False))\n        observations[\'four_bit_model_loaded\'] = loaded\n        if not loaded:\n            raise ValueError(\'Expected a 4-bit QLoRA base model\')\n        identity = runtime.identity()\n        observations[\'model_identity\'] = identity\n        if identity[\'model_revision\'] != REVISION:\n            raise ValueError(\'Loaded model revision differs from pinned job\')\n        return value\n\n    def observe_restore(runtime, checkpoint):\n        value = original_restore(runtime, checkpoint)\n        observations[\'adapter_restores\'].append({\'path\': str(checkpoint), \'files\': adapter_files(checkpoint)})\n        return value\n\n    def observe_train(trainer, *a, **kw):\n        value = original_train(trainer, *a, **kw)\n        observations[\'optimizer_steps\'] = trainer.state.global_step\n        observations[\'training_log_history\'] = trainer.state.log_history\n        observations[\'training_loss\'] = float(value.training_loss)\n        losses = [x[\'loss\'] for x in trainer.state.log_history if \'loss\' in x]\n        if trainer.state.global_step != 2 or len(losses) != 2 or not all(math.isfinite(x) for x in [value.training_loss, *losses]):\n            raise ValueError(\'Expected exactly two optimizer steps with two finite logged losses\')\n        return value\n\n    def observe_infer(runtime, payload):\n        value = original_infer(runtime, payload)\n        observations[\'inference\'] = {\'payload\': payload, \'prediction\': value}\n        return value\n\n    StudentRuntime.prepare, StudentRuntime.restore, StudentRuntime.infer = observe_prepare, observe_restore, observe_infer\n    trl.SFTTrainer.train = observe_train\n    started = time.monotonic()\n    previous = None\n    original_setitimer = signal.setitimer\n    torch.cuda.reset_peak_memory_stats(0)\n    try:\n        if args.billing_mode == \'paid\':\n            # Preserve the existing worker\'s own cumulative runtime-cost watchdog.\n            def bounded_timer(which, seconds, interval=0):\n                return original_setitimer(which, min(seconds, args.max_runtime_seconds) if seconds > 0 else 0, interval)\n            signal.setitimer = bounded_timer\n            result = gpu_worker.run(root, authorized=True, max_gpu_usd=args.max_gpu_usd,\n                                    gpu_hourly_rate_usd=args.gpu_hourly_rate_usd)\n        else:\n            def timeout(signum, frame):\n                raise TimeoutError(\'Free GPU canary wall-clock limit exhausted\')\n            previous = signal.signal(signal.SIGALRM, timeout)\n            signal.setitimer(signal.ITIMER_REAL, args.max_runtime_seconds)\n            result = gpu_worker._run(root, authorized=True)\n            elapsed = time.monotonic() - started\n            if elapsed > args.max_runtime_seconds:\n                raise TimeoutError(\'Free GPU canary exceeded wall-clock limit\')\n            result.update(gpu_runtime_seconds=elapsed, estimated_gpu_cost_usd=0.0,\n                          gpu_budget_usd=args.max_gpu_usd, gpu_hourly_rate_usd=0.0,\n                          gpu_cost_limitation=\'Operator confirmed free allocation; actual usage charge is zero. No equivalent commercial GPU cost was invented.\')\n        saved = root / job[\'output_directory\']\n        files = adapter_files(saved / \'adapter\')\n        if files != result[\'adapter_files\'] or any((saved / \'adapter\' / name).stat().st_size == 0 for name in files):\n            raise ValueError(\'Saved adapter is empty or checksum verification failed\')\n        if len(observations[\'adapter_restores\']) != 2 or observations[\'adapter_restores\'][0][\'files\'] != ADAPTER or observations[\'adapter_restores\'][1][\'files\'] != files:\n            raise ValueError(\'Starting/saved adapter reload evidence is incomplete\')\n        verify(root)  # Initial adapter, dataset and packaged source remain unchanged.\n        observations.update(status=\'completed\', infrastructure_canary_passed=True)\n        result[\'canary_observations\'] = observations\n        atomic_json(saved / \'result.json\', result)\n        return result\n    except BaseException:\n        observations.update(status=\'failed\', infrastructure_canary_passed=False, failure_traceback=traceback.format_exc())\n        # Never leave an importable success receipt after an observation failure.\n        success = root / \'result/result.json\'\n        if success.exists():\n            success.rename(root / \'result/unverified-worker-result.json\')\n        raise\n    finally:\n        signal.setitimer = original_setitimer\n        if previous is not None:\n            signal.setitimer(signal.ITIMER_REAL, 0)\n            signal.signal(signal.SIGALRM, previous)\n        observations[\'total_wall_seconds\'] = time.monotonic() - started\n        try:\n            observations.update(peak_allocated_vram_bytes=torch.cuda.max_memory_allocated(0),\n                                peak_reserved_vram_bytes=torch.cuda.max_memory_reserved(0))\n        except Exception as exc:\n            observations[\'memory_measurement_error\'] = str(exc)\n        save(path, observations)\n        if observations[\'status\'] == \'completed\':\n            save(root / \'result/canary-observations.json\', observations)\n            result[\'canary_observations\'] = observations\n            atomic_json(root / \'result/result.json\', result)\n        trl.SFTTrainer.train = original_train\n        StudentRuntime.prepare, StudentRuntime.restore, StudentRuntime.infer = original_prepare, original_restore, original_infer\n\n\ndef export(root):\n    destination = root.parent / \'test-b-return.zip\'\n    if destination.exists():\n        raise ValueError(\'Return archive already exists; preserve the earlier download\')\n    files = [p for p in (root / \'result\').rglob(\'*\') if p.is_file()]\n    files += [p for p in root.parent.iterdir() if p.is_file() and p.suffix in (\'.json\', \'.log\')]\n    if not files:\n        raise ValueError(\'No results or diagnostic logs to export\')\n    with zipfile.ZipFile(destination, \'w\', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as z:\n        for p in files:\n            name = \'result/\' + str(p.relative_to(root / \'result\')) if p.is_relative_to(root / \'result\') else \'diagnostics/\' + p.name\n            z.write(p, name)\n    checksum = sha(destination)\n    destination.with_suffix(\'.zip.sha256\').write_text(checksum + \'  \' + destination.name + \'\\n\')\n    return {\'archive\': str(destination), \'sha256\': checksum,\n            \'import_only_if\': \'result/result.json exists and training-observations.json status is completed\'}\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'phase\', choices=(\'extract\', \'verify\', \'preflight\', \'train\', \'export\'))\n    p.add_argument(\'--root\', type=Path, required=True)\n    p.add_argument(\'--archive\', type=Path)\n    p.add_argument(\'--authorize-training\', action=\'store_true\')\n    p.add_argument(\'--provider\')\n    p.add_argument(\'--expected-gpu\')\n    p.add_argument(\'--billing-mode\', choices=(\'free\', \'paid\'))\n    p.add_argument(\'--max-gpu-usd\', type=float, default=1.0)\n    p.add_argument(\'--gpu-hourly-rate-usd\', type=float, default=float(\'nan\'))\n    p.add_argument(\'--max-runtime-seconds\', type=float, default=1200)\n    a = p.parse_args()\n    root = a.root.resolve()\n    os.environ.setdefault(\'CUDA_VISIBLE_DEVICES\', \'0\')\n    if a.phase == \'extract\':\n        value = extract(a.archive, root)\n    elif a.phase == \'verify\':\n        value = verify(root)\n    elif a.phase == \'preflight\':\n        value = preflight(root)\n    elif a.phase == \'train\':\n        value = train(root, a)\n    else:\n        value = export(root)\n    print(json.dumps(value, indent=2, allow_nan=False))\n\n\nif __name__ == \'__main__\':\n    main()\n')
assert hashlib.sha256(RUNNER.read_bytes()).hexdigest() == '5c38c190d5b31d689db77c28a4d7176b8f24921287fc4f732c998dd30fcb1634'
print("Runner ready; no training started.")


## 2. Upload and verify the existing package
Colab will open a file picker: choose the original `package.zip` from the SSD.
Kaggle: attach the ZIP as an input dataset, then paste its `/kaggle/input/.../package.zip` path.
Keep this project’s adapter/data private when creating the input dataset.


In [ ]:
if Path('/content').exists() and not Path('/kaggle/working').exists():
    from google.colab import files
    previous = Path.cwd()
    os.chdir(BASE)
    try: uploaded = files.upload()
    finally: os.chdir(previous)
    if len(uploaded) != 1:
        raise RuntimeError('Upload exactly the original package ZIP.')
    ARCHIVE = BASE / next(iter(uploaded))
    del uploaded
else:
    ARCHIVE = Path(input('Uploaded original package ZIP path: ').strip())
execute([sys.executable, RUNNER, 'extract', '--archive', ARCHIVE, '--root', ROOT], 'extract.log')
print('Original package hash and every source, dataset and adapter hash verified.')


## 3. Install isolated CUDA/PyTorch and packaged project dependencies
This does not load model weights. Allow at least 40 GiB free disk for dependencies, model cache and outputs.
Python 3.13+ notebook kernels are supported: the cell first looks for an installed Python 3.10–3.12.
If none exists, it installs uv 0.9.11 locally and downloads Python 3.12.11 for the training environment.
This uses [uv's documented Python installation and environment support](https://docs.astral.sh/uv/guides/install-python/).
The CUDA wheel needs a compatible host NVIDIA driver. Driver failures must be fixed before training.
Pinning is based on the [official PyTorch 2.8 CUDA 12.6 wheel instructions](https://pytorch.org/get-started/previous-versions/)
and [bitsandbytes 0.48.2 compatibility](https://huggingface.co/docs/bitsandbytes/v0.48.2/installation).
The complete combination still requires live validation on your allocated GPU.


In [ ]:
if shutil.disk_usage(BASE).free < 40 * 1024**3:
    raise RuntimeError('Need at least 40 GiB free cloud-local disk; no model has been loaded.')
if VENV.exists():
    raise RuntimeError('Environment already exists; inspect installation logs instead of reinstalling automatically.')
training_python = None
candidates = [sys.executable] + [shutil.which(name) for name in ('python3.12', 'python3.11', 'python3.10')]
for candidate in dict.fromkeys(x for x in candidates if x):
    version = tuple(json.loads(subprocess.check_output(
        [candidate, '-c', 'import json,sys; print(json.dumps(list(sys.version_info[:2])))'], text=True)))
    if (3, 10) <= version <= (3, 12):
        training_python = candidate
        break
if training_python:
    execute([training_python, '-m', 'venv', VENV], 'venv.log')
else:
    bootstrap = BASE / 'uv-bootstrap'
    if bootstrap.exists():
        raise RuntimeError('Previous bootstrap attempt exists; inspect its logs before retrying.')
    execute([sys.executable, '-m', 'pip', 'install', '--no-deps', '--target', bootstrap,
             'uv==0.9.11'], 'uv-bootstrap.log')
    uv = bootstrap / 'bin/uv'
    if not uv.is_file():
        raise RuntimeError('Pinned uv executable missing; inspect uv-bootstrap.log')
    os.environ['UV_PYTHON_INSTALL_DIR'] = str(BASE / 'managed-python')
    os.environ['UV_CACHE_DIR'] = str(BASE / 'uv-cache')
    execute([uv, 'python', 'install', '3.12.11'], 'python-install.log')
    execute([uv, 'venv', '--python', '3.12.11', '--seed', VENV], 'venv.log')
training_version = json.loads(subprocess.check_output(
    [PY, '-c', 'import json,sys; print(json.dumps(list(sys.version_info[:3])))'], text=True))
if not (3, 10) <= tuple(training_version[:2]) <= (3, 12):
    raise RuntimeError(f'Training interpreter is incompatible: {training_version}')
(BASE / 'python-environment.json').write_text(json.dumps(
    {'notebook_host': host, 'training_python': str(PY), 'training_version': training_version}, indent=2))
print('Isolated training Python:', training_version)
constraints = BASE / 'constraints.txt'
constraints.write_text('torch==2.8.0\ntorchvision==0.23.0\ntorchaudio==2.8.0\ntransformers==4.57.3\ntrl==0.25.1\npeft==0.18.0\nlighteval==0.11.0\naccelerate==1.11.0\nbitsandbytes==0.48.2\ndatasets==4.4.1\ntokenizers==0.22.2\nhuggingface-hub==0.36.0\nsafetensors==0.6.2\n')
execute([PY, '-m', 'pip', 'install', '--upgrade', 'pip'], 'pip-bootstrap.log')
execute([PY, '-m', 'pip', 'install', '--no-cache-dir', 'torch==2.8.0', 'torchvision==0.23.0',
         'torchaudio==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126'], 'pytorch-install.log')
execute([PY, '-m', 'pip', 'install', '--no-cache-dir', '-c', constraints,
         '-r', ROOT / 'requirements-training.txt', 'tokenizers==0.22.2',
         'huggingface-hub==0.36.0', 'safetensors==0.6.2'], 'dependencies-install.log')
execute([PY, '-m', 'pip', 'check'], 'pip-check.log')
execute([PY, '-m', 'pip', 'freeze'], 'pip-freeze.log')


## 4. Check GPU and run the original worker’s validate-only mode
Verifies package integrity, all 85 targets, tokenizer/context, non-thinking template,
actual TRL completion-only masks and zero truncation. No full model loading or training.
Inspect `cloud-preflight.json` and both validate-only logs if anything fails.


In [ ]:
execute([PY, RUNNER, 'preflight', '--root', ROOT], 'preflight.log')
PREFLIGHT = json.loads((BASE / 'cloud-preflight.json').read_text())
GPU = PREFLIGHT['environment']['gpu']
print('Detected GPU:', GPU)
print('Maximum sequence:', PREFLIGHT['tokenizer']['sequence_tokens']['max'])
print('Overlength:', PREFLIGHT['tokenizer']['overlength_examples'])
print('STOP HERE until you approve the actual allocation and two-step run.')


## 5. Explicitly authorize this allocation and run exactly two steps
Do not run this cell until preflight passes. Enter the actual provider and billing status.
For free Colab/Kaggle, confirm the specific allocation costs $0; no invented comparison hourly rate is used.
Paid mode requires the actual approved positive hourly rate and a budget no greater than $1.
Colab compute-unit pricing that cannot be expressed as a known USD/hour rate is not supported in paid mode.
The worker watchdog covers this worker only; it cannot stop provider idle charges. Shut down paid hardware yourself.
An additional 1,200-second cap can only tighten the worker limit. No retries, step/context increases or new adapters.
The wrapper records actual steps/losses, 4-bit loading, exact adapter restores, inference text and peak PyTorch VRAM.
Free mode calls the unchanged worker core with an explicit wall-clock watchdog because the packaged paid CLI rejects rate zero.


In [ ]:
if (ROOT / 'result').exists() or (BASE / 'training-observations.json').exists():
    raise RuntimeError('Previous training attempt exists. Export it; do not retrain automatically.')
provider = input('Actual GPU provider (Colab/Kaggle/other approved provider): ').strip()
billing = input('Actual billing mode: type free or paid: ').strip().lower()
if billing not in ('free', 'paid'): raise RuntimeError('Unknown billing mode')
if billing == 'free':
    if input('Type FREE to confirm this allocation has no usage charge: ').strip() != 'FREE':
        raise RuntimeError('Free allocation not confirmed')
    rate, budget = '0', '1'
else:
    rate = input('Actual approved GPU hourly rate in USD: ').strip()
    budget = input('Approved worker budget in USD, at most 1: ').strip()
statement = f'APPROVE {provider} | {GPU} | {rate} USD/hour | {budget} USD cap | 2 steps'
print('Review and type exactly:', statement)
if input().strip() != statement:
    raise RuntimeError('Training not authorized; no model loaded')
execute([PY, RUNNER, 'train', '--root', ROOT, '--authorize-training', '--provider', provider,
         '--expected-gpu', GPU, '--billing-mode', billing, '--gpu-hourly-rate-usd', rate,
         '--max-gpu-usd', budget, '--max-runtime-seconds', '1200'], 'training.log')
outcome = json.loads((ROOT / 'result/result.json').read_text())
print('Runtime seconds:', outcome['gpu_runtime_seconds'])
print('GPU cost USD:', outcome['estimated_gpu_cost_usd'])
print('Observations:', json.dumps(outcome['canary_observations'], indent=2))


## 6. Export success results or exact failure diagnostics
Run this cell after success OR failure. Download the ZIP and SHA-256 sidecar before shutting down.
A failed run is diagnostic only and must not be imported as a completed checkpoint.


In [ ]:
execute([PY if PY.exists() else sys.executable, RUNNER, 'export', '--root', ROOT], 'export.log')
archive = BASE / 'test-b-return.zip'
sidecar = BASE / 'test-b-return.zip.sha256'
print(sidecar.read_text())
if Path('/content').exists() and not Path('/kaggle/working').exists():
    from google.colab import files
    files.download(str(archive)); files.download(str(sidecar))
else:
    from IPython.display import FileLink, display
    display(FileLink(str(archive))); display(FileLink(str(sidecar)))
    print('Also available under Kaggle /kaggle/working/test-b/. Download before ending the session.')


## Return to the Mac
Save both downloads on the external SSD under `experiments/gpu-canary-20261119/test-b/returned/`.
Verify the ZIP hash against the sidecar, extract it, and inspect `result/result.json` and
`diagnostics/training-observations.json` before import. The accompanying runbook gives exact Mac commands.
Preserve the original adapter. Do not promote this two-step engineering checkpoint or claim model improvement.
